# Using SELMR

## Introduction

A *Simple Explainable Language Multiset Representation* (SELMR) is a text data structure that works like a language model. The SELMR data structure consists of multisets created from all phrases (i.e. multiword expressions) and all phrase-context combinations contained in a collection of documents that satisfy some contraints. The multisets can be used for downstream NLP tasks like text classifications and searching, similar to real-valued vector embeddings.

SELMRs produce explainable results without any randomness and enable explicit links with lexical, linguistical and terminological annotations. No model is trained to get real-valued vector embeddings and no dimensionality reduction is applied.

The building blocks for SELMR are phrases and contexts.

* A phrase can be any list of consecutive words that occurs in a collection of documents, with a certain maximum length and a certain minimum number of occurrences.

* A context of a phrase is the combination of the preceding list of words (the left side) and the following list of words (the right side) of that phrase, with a certain maximum length and a certain minimum number of occurrences.

Of each phrase the SELMR data structure contains the contexts in which the phrase occurs with the number of that phrase-context combinations in the documents (forming a multiset or a collections.Counter in Python). Of each context the SELMR data structure contains the multiset with the phrases that occur in the context with their respective number of occurrences in the documents.

In [2]:
documents = [
    "We walked in the beautiful park.",
    "Then we did some shopping in the city."
]

In [3]:
from selmr import SELMR

# Create a SELMR data structure given the two sentences above
selmr = SELMR(
    documents=documents
)

In [4]:
selmr.contexts("city")

Counter({('the', '.'): 1, ('in the', '.'): 1, ('shopping in the', '.'): 1})

```console
Counter({('the', '.'): 1, ('in the', '.'): 1, ('shopping in the', '.'): 1})
```

In [5]:
selmr.contexts("beautiful park")

Counter({('the', '.'): 1, ('in the', '.'): 1, ('walked in the', '.'): 1})

```console
Counter({('the', '.'): 1, ('in the', '.'): 1, ('walked in the', '.'): 1})
```

In [6]:
selmr.most_similar("city")

Counter({'city': 3, 'beautiful park': 2})

```console
Counter({'city': 3, 'beautiful park': 2})
```

## SELMR based on DBpedia

These are results of a SELMR created with 10.000 DBpedia pages. We defined a context of a word in it simplest form: the tuple of the previous multiwords and the next multiwords (no preprocessing, no changes to the text, i.e. no deletion of stopwords and punctuation). The maximum phrase length is five words, the maximum left and right context length is also five words.

In [7]:
import pickle
with open('..//data//dbpedia_phrases_10000.pickle', 'rb') as handle:
    phrases = pickle.load(handle)
with open('..//data//dbpedia_contexts_10000.pickle', 'rb') as handle:
    contexts = pickle.load(handle)

In [8]:
from selmr import SELMR, LanguageMultisets

# construct a SELMR data structure with the DBpedia phrases and contexts
selmr = SELMR(
    multisets=LanguageMultisets(phrases, contexts, None)
)

### Most frequent contexts of a phrase

The ten most frequent contexts in which the word 'has' occurs with their number of occurrences are the following:

In [9]:
# most frequent contexts of the word "has"
selmr.contexts("has", topn=10)

Counter({('It', 'been'): 2014,
         ('it', 'been'): 1970,
         ('SENTSTART It', 'been'): 1858,
         ('and', 'been'): 1201,
         ('which', 'been'): 987,
         ('that', 'been'): 813,
         ('and', 'a'): 806,
         ('also', 'a'): 774,
         ('there', 'been'): 764,
         ('which', 'a'): 624})

This results in

```console
Counter({('It', 'been'): 1031,
         ('SENTSTART It', 'been'): 951,
         ('it', 'been'): 836,
         ('and', 'been'): 642,
         ('which', 'been'): 521,
         ('also', 'a'): 436,
         ('there', 'been'): 420,
         ('and', 'a'): 418,
         ('that', 'been'): 339,
         ('it', 'a'): 270})
```

This means that the corpus contains ... occurrences of 'It has been', i.e. occurrences where the word 'has' occurred in the context ('It', 'been').

SENTSTART and SENTEND are tokens to indicate the start and end of a sentence. This makes it possible to derive the contexts of a phrase that starts or ends a sentence. 

### Phrase and context frequencies

The contexts in which a word occurs represent to some extent the properties and the meaning of a word. If you derive the phrases that share the most frequent contexts of the word 'has' then you get the following table (the columns contains the contexts, the rows the phrases that have the most contexts in common):

In [10]:
import pandas as pd
pd.DataFrame().from_dict(
    selmr.dict_phrases_contexts("has", topcontexts=10), orient='tight'
)

left context,It,it,SENTSTART It,and,which,that,and,also,there,which
right context,been,been,been,been,been,been,a,a,been,a
phrase,,,,,,,,,,
would have,26,156,25,19,110,97,2,2,31,4
had,139,815,130,327,1696,1388,623,524,350,306
has,2014,1970,1858,1201,987,813,806,774,764,624
may have,48,151,48,113,146,85,6,0,60,6
have,0,19,0,412,477,942,370,299,773,185
has not,14,104,14,34,27,47,0,0,10,0
could have,2,42,2,4,17,40,0,0,7,0


This results in:

```console

                  It 	it      SENTSTART It    and     which   also    there   and
                  been 	been 	been 	        been    been    a       been    a

has               1031  1021    954             642   521     436     420     418
had               71    402     53              169   886     266     171     336
would have        14 	75   	10              9     53      2       15 	2
may have          26 	82  	26              37    61      0       33 	2
could have        2 	24  	2               2     7       0       4 	0
has also          149 	60  	180             12    5       0       11 	0
has always        2 	3   	2               2     3       0       2 	0

```

The contexts that a word has in common with contexts of another word can be used as a measure of similarity. The word 'had' (second row) has eight contexts in common with the word 'has' so this word is very similar. The phrase 'would have' (seventh row) has seven contexts in common, so 'would have' is also similar but less similar than the word 'had'. We used a limited number of contexts to show the idea; normally a higher number of contexts can be used to compare the similarity of words.

The word similarities found can in this case explained as follows. Similar words are forms of the verb 'have'. This is because the verb is often used in the construction of perfect tenses where the verb 'have' is combined with the past participle of another verb, in this case the often occuring 'been'. Note that the list contains 'has not'.

### Phrase similarities

Based on the approach above we can derive top phrase similarities.

In [11]:
# top phrase similarities of the word "has"
selmr.most_similar("has", topn=10, topcontexts=15)

Counter({'had': 15,
         'has': 15,
         'would have': 11,
         'may have': 10,
         'have': 9,
         'is': 8,
         'was': 8,
         'has not': 7,
         'could have': 7,
         'also has': 7})

This results in

```console
{'had': (15, 15),
 'has': (15, 15),
 'would have': (12, 15),
 'have': (10, 15),
 'may have': (10, 15),
 'could have': (9, 15),
 'has never': (8, 15),
 'has not': (8, 15),
 'also has': (7, 15),
 'had long': (7, 15)}
```

Now take a look at similar words of 'larger'.

In [12]:
# top phrase similarities of the word "larger"
selmr.most_similar("larger", topn=10, topcontexts=15)

Counter({'larger': 15,
         'smaller': 14,
         'higher': 13,
         'greater': 13,
         'longer': 11,
         'faster': 11,
         'less': 11,
         'lower': 10,
         'shorter': 10,
         'better': 10})

Resulting in:

```console
{'larger': (15, 15),
 'smaller': (14, 15),
 'greater': (13, 15),
 'higher': (12, 15),
 'longer': (11, 15),
 'better': (10, 15),
 'faster': (10, 15),
 'less': (10, 15),
 'lower': (10, 15),
 'shorter': (10, 15)}
```

Like the word 'larger' these are all comparative adjectives. These words are similar because they share the most frequent contexts, in this case contexts like (is, than) and (much, than).

In [13]:
# top phrase similarities of the word "might"
selmr.most_similar("might", topn=10, topcontexts=25)

Counter({'could': 25,
         'would': 25,
         'might': 25,
         'may': 25,
         'should': 25,
         'must': 24,
         'would not': 22,
         'will': 22,
         'can': 21,
         'may not': 21})

```console
{'could': (25, 25),
 'may': (25, 25),
 'might': (25, 25),
 'should': (25, 25),
 'would': (25, 25),
 'must': (24, 25),
 'would not': (23, 25),
 'could not': (22, 25),
 'will': (21, 25),
 'can': (20, 25)}
```

Most frequent coinciding contexts are in this case ('it', 'be'), ('he', 'have') and ('that', 'be').

Contexts can also be used to find 'semantic' similarities.

In [14]:
# top phrase similarities of the word "King"
selmr.most_similar("King", topn=10, topcontexts=25)

Counter({'King': 25,
         'Prince': 18,
         'Emperor': 16,
         'Duke': 15,
         'king': 14,
         'President': 14,
         'president': 13,
         'one': 12,
         'Queen': 12,
         'Prime Minister': 11})

This results in

```console
{'King': (15, 15),
 'President': (8, 15),
 'Queen': (8, 15),
 'king': (8, 15),
 'Emperor': (7, 15),
 'Kingdom': (7, 15),
 'Prince': (7, 15),
 'enemies': (7, 15),
 'kings': (7, 15),
 'president': (7, 15)}
```


Instead of single words we can also find the similarities of multiwords

In [15]:
# top phrase similarities of Barack Obama
selmr.most_similar("Barack Obama", topn=10, topcontexts=15)

Counter({'Barack Obama': 15,
         'Ronald Reagan': 6,
         'Donald Trump': 5,
         'Lyndon B Johnson': 4,
         'George W Bush': 4,
         'of the United States': 4,
         'Bush': 4,
         'Kabbah': 3,
         'Franklin D Roosevelt': 3,
         'George H W Bush': 3})

```console
{'Barack Obama': (15, 15),
 'Bill Clinton': (5, 15),
 'Ronald Reagan': (5, 15),
 'Franklin D Roosevelt': (4, 15),
 'George W Bush': (4, 15),
 'Richard Nixon': (4, 15),
 'Bush': (3, 15),
 'Dwight D Eisenhower': (3, 15),
 'George H W Bush': (3, 15)}
```


### Most frequent phrases of a context

Here are some examples of the most frequent phrases of a context.

In [16]:
context = ("King", "of England")
for r in selmr.phrases(context, topn=10).items():
    print(r)

('Charles II', 59)
('Charles I', 42)
('James I', 38)
('Henry VIII', 34)
('Edward I', 27)
('James II', 25)
('Henry VII', 24)
('John', 20)
('Henry III', 19)
('Edward III', 18)


```console
('Henry VIII', 15)
('Charles II', 12)
('John', 12)
('Henry III', 8)
('James I', 8)
('Edward I', 7)
('Edward III', 6)
('Charles I', 5)
('Henry VII', 5)
('Henry II', 4)
```

In [17]:
context = ("the", "city")
for r in selmr.phrases(context, topn=10).items():
    print(r)

('capital', 355)
('largest', 266)
('inner', 120)
('old', 114)
('ancient', 92)
('first', 91)
('second largest', 88)
('capital and largest', 84)
('host', 66)
('centre of the', 60)


```console
('capital', 141)
('largest', 140)
('old', 55)
('inner', 52)
('first', 48)
('second largest', 44)
('ancient', 43)
('most populous', 39)
('Greek', 37)
('port', 31)
```

In [18]:
context = ("he", "that")
for r in selmr.phrases(context, topn=10).items():
    print(r)

('believed', 244)
('said', 207)
('stated', 194)
('argued', 148)
('felt', 135)
('wrote', 115)
('argues', 108)
('found', 97)
('noted', 89)
('claimed', 88)


### Phrase similarities given a specific context

Some phrases have multiple meanings. Take a look at the contexts of the word 'deal':

In [19]:
selmr.contexts("deal", topn=10)

Counter({('to', 'with'): 946,
         ('great', 'of'): 649,
         ('a great', 'of'): 610,
         ('to', 'with the'): 332,
         ('a', 'with'): 225,
         ('good', 'of'): 86,
         ('a good', 'of'): 83,
         ('had to', 'with'): 71,
         ('the', 'SENTEND'): 57,
         ('a', 'to'): 53})

This results in:

```console
Counter({('to', 'with'): 487,
         ('great', 'of'): 348,
         ('a great', 'of'): 326,
         ('to', 'with the'): 165,
         ('a', 'with'): 84,
         ('a good', 'of'): 36,
         ('had to', 'with'): 35,
         ('good', 'of'): 28,
         ('SENTSTART The', 'was'): 25,
         ('The', 'was'): 25})
```


In some of these contexts 'deal' is a verb meaning 'to do business' and in other contexts 'deal' is a noun meaning a 'contract' or an 'agreement'. The specific meaning can be derived from the context in which the phrase is used.

It is possible to take into account a specific context when using the most_similar function in the following way:

In [20]:
selmr.most_similar(phrase="deal", context=("to", "with"), topcontexts=50, topphrases=15, topn=10)

Counter({'deal': 50,
         'work': 21,
         'compete': 10,
         'comply': 8,
         'cope': 8,
         'communicate': 7,
         'do': 6,
         'live': 6,
         'meet': 5,
         'be confused': 4})

The result is:

```console
{'deal': (50, 50),
 'work': (20, 50),
 'comply': (9, 50),
 'compete': (7, 50),
 'cope': (7, 50),
 'interact': (7, 50),
 'coincide': (6, 50),
 'communicate': (6, 50),
 'do': (6, 50),
 'help': (6, 50)}
```

So these are all verbs, similar to the verb 'deal'.

In [21]:
selmr.most_similar(phrase="deal", context=("a", "with"), topcontexts=100, topphrases=100, topn=10)

Counter({'deal': 100,
         'contract': 24,
         'treaty': 19,
         'meeting': 17,
         'man': 16,
         'relationship': 15,
         'person': 14,
         'partnership': 9,
         'collaboration': 4,
         'joint venture': 3})

In this case the result is:

```console
{'deal': (50, 50),
 'contract': (12, 50),
 'treaty': (12, 50),
 'meeting': (11, 50),
 'relationship': (11, 50),
 'man': (10, 50),
 'dispute': (8, 50),
 'partnership': (8, 50),
 'person': (8, 50),
 'coalition': (7, 50)}
```

So, now the results are nouns, and similar to the noun 'deal'.

### Phrase similarities given a set of contexts

If you want to find the phrases that fit a set of contexts then this is also possible.

In [22]:
c1 = [
        c[0] for c in (
            selmr.contexts("considered", topn=None) &
            selmr.contexts("believed", topn=None)
         ).most_common(15)
]

This results in:

```console
[('is', 'to'),
 ('is', 'to be'),
 ('are', 'to'),
 ('was', 'to'),
 ('are', 'to be'),
 ('is', 'to have'),
 ('is', 'to be the'),
 ('was', 'to be'),
 ('were', 'to'),
 ('generally', 'to'),
 ('are', 'to have'),
 ('is', 'to be a'),
 ('is', 'by'),
 ('widely', 'to'),
 ('he', 'to')]
```

In [23]:
selmr.most_similar(contexts=c1, topn=10)

TypeError: SELMR.most_similar() got an unexpected keyword argument 'contexts'

Resulting in:

```console
{'believed': (15, 15),
 'considered': (15, 15),
 'thought': (14, 15),
 'expected': (13, 15),
 'known': (13, 15),
 'reported': (13, 15),
 'said': (13, 15),
 'assumed': (12, 15),
 'claimed': (12, 15),
 'held': (12, 15)}
```